# Notebook 23 — Predictive Constraint Routing

Use decompression forecasts to route before fallback occurs. This notebook follows the Notebook 21/22 artifact format and writes linked report artifacts using `figures/` paths.


In [ ]:

# ============================================================
# Notebook 23
# Predictive Constraint Routing
# Constraint-Guided Coherence Score (CGCS)
# ============================================================

# ============================================================
# 00. Imports
# ============================================================

import os
import json
import zipfile
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
from sklearn.model_selection import train_test_split


# ============================================================
# 01. Directories
# ============================================================

os.makedirs("results", exist_ok=True)
os.makedirs("figures", exist_ok=True)
os.makedirs("reports", exist_ok=True)


# ============================================================
# 02. Synthetic compressed-routing stream
# ============================================================

np.random.seed(43)

N = 240
windows = np.arange(N)

macro_routes = np.random.choice(
    ["macro_0", "macro_1", "macro_2", "macro_3", "macro_4"],
    size=N,
    p=[0.34, 0.12, 0.20, 0.18, 0.16],
)

# Coherence has ordinary variation plus a coherent plateau and two instability bands.
macro_cgcs_score = (
    0.55
    + 0.08 * np.sin(np.linspace(0, 8 * np.pi, N))
    + np.random.normal(0, 0.08, N)
)

macro_cgcs_score[106:145] += 0.13
macro_cgcs_score[35:55] -= 0.12
macro_cgcs_score[205:230] -= 0.10
macro_cgcs_score = np.clip(macro_cgcs_score, 0.20, 0.84)

rolling_stability = (
    pd.Series(macro_cgcs_score)
    .rolling(12, min_periods=1)
    .mean()
)

rolling_volatility = (
    pd.Series(macro_cgcs_score)
    .rolling(10, min_periods=1)
    .std()
    .fillna(0)
)

rolling_switch_rate = (
    pd.Series(macro_routes)
    .ne(pd.Series(macro_routes).shift())
    .rolling(15, min_periods=1)
    .mean()
)

rolling_residual = 1 - rolling_stability + rolling_volatility

rolling_pressure_raw = (
    0.40 * rolling_switch_rate
    + 0.35 * rolling_volatility
    + 0.25 * rolling_residual
)

rolling_pressure = (
    (rolling_pressure_raw - rolling_pressure_raw.min())
    / (rolling_pressure_raw.max() - rolling_pressure_raw.min())
)

route_encoding = {r: i for i, r in enumerate(sorted(set(macro_routes)))}
macro_route_id = pd.Series(macro_routes).map(route_encoding).to_numpy()


# ============================================================
# 03. Decompression target from Notebook 22-style forecasting
# ============================================================

future_horizon = 5

decompression_event = (
    (macro_cgcs_score < 0.45)
    & (rolling_pressure > 0.55)
).astype(int)

future_decompression = (
    pd.Series(decompression_event)
    .rolling(future_horizon, min_periods=1)
    .max()
    .shift(-future_horizon)
    .fillna(0)
    .astype(int)
)

stream = pd.DataFrame({
    "window": windows,
    "macro_route": macro_routes,
    "macro_route_id": macro_route_id,
    "macro_cgcs_score": macro_cgcs_score,
    "rolling_stability": rolling_stability,
    "rolling_volatility": rolling_volatility,
    "rolling_switch_rate": rolling_switch_rate,
    "rolling_residual": rolling_residual,
    "rolling_pressure": rolling_pressure,
    "decompression_event": decompression_event,
    "future_decompression": future_decompression,
})

feature_cols = [
    "macro_cgcs_score",
    "rolling_stability",
    "rolling_volatility",
    "rolling_switch_rate",
    "rolling_residual",
    "rolling_pressure",
    "macro_route_id",
]

X = stream[feature_cols]
y = stream["future_decompression"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=43,
    stratify=y,
)

forecast_model = RandomForestClassifier(
    n_estimators=250,
    max_depth=6,
    min_samples_leaf=3,
    random_state=43,
)

forecast_model.fit(X_train, y_train)

stream["decompression_probability"] = forecast_model.predict_proba(X)[:, 1]
stream["forecast_decompression"] = (
    stream["decompression_probability"] >= 0.50
).astype(int)

auc = roc_auc_score(y, stream["decompression_probability"])


# ============================================================
# 04. Reactive vs predictive routing policies
# ============================================================

# Reactive routing only falls back once an observed decompression event occurs.
reactive_gate = np.where(
    stream["decompression_event"] == 1,
    "fallback",
    np.where(stream["macro_cgcs_score"] >= 0.70, "accepted", "watch"),
)

# Predictive routing acts before collapse when forecast risk is high.
# It distinguishes stabilizing reroute from hard fallback.
predictive_gate = []
predictive_action = []

for _, row in stream.iterrows():
    risk = row["decompression_probability"]
    cgcs = row["macro_cgcs_score"]
    pressure = row["rolling_pressure"]

    if risk >= 0.70 and pressure >= 0.55:
        predictive_gate.append("fallback")
        predictive_action.append("protective_fallback")
    elif risk >= 0.50:
        predictive_gate.append("reroute")
        predictive_action.append("predictive_reroute")
    elif cgcs >= 0.70:
        predictive_gate.append("accepted")
        predictive_action.append("retain_compressed")
    else:
        predictive_gate.append("watch")
        predictive_action.append("monitor")

stream["reactive_gate"] = reactive_gate
stream["predictive_gate"] = predictive_gate
stream["predictive_action"] = predictive_action

# Predictive routing is credited when it reroutes before a future decompression label.
stream["forecast_hit"] = (
    (stream["forecast_decompression"] == 1)
    & (stream["future_decompression"] == 1)
).astype(int)

stream["avoidable_fallback"] = (
    (stream["future_decompression"] == 1)
    & (stream["predictive_gate"].isin(["reroute", "fallback"]))
).astype(int)

# Stability proxy: predictive routing improves stability when forecast pressure is handled
# by reroute/fallback before actual decompression.
reactive_stability = np.clip(
    stream["rolling_stability"]
    - 0.22 * (stream["reactive_gate"] == "fallback").astype(float)
    - 0.10 * stream["future_decompression"],
    0,
    1,
)

predictive_stability = np.clip(
    stream["rolling_stability"]
    + 0.12 * (stream["predictive_gate"] == "reroute").astype(float)
    + 0.06 * (stream["predictive_gate"] == "accepted").astype(float)
    - 0.11 * (stream["predictive_gate"] == "fallback").astype(float)
    - 0.04 * stream["future_decompression"],
    0,
    1,
)

stream["reactive_stability"] = reactive_stability
stream["predictive_stability"] = predictive_stability
stream["stability_gain"] = predictive_stability - reactive_stability

stream["reactive_switch"] = (
    pd.Series(stream["reactive_gate"])
    .ne(pd.Series(stream["reactive_gate"]).shift())
    .astype(int)
)

stream["predictive_switch"] = (
    pd.Series(stream["predictive_gate"])
    .ne(pd.Series(stream["predictive_gate"]).shift())
    .astype(int)
)

stream["reactive_switch_rate"] = (
    stream["reactive_switch"]
    .rolling(15, min_periods=1)
    .mean()
)

stream["predictive_switch_rate"] = (
    stream["predictive_switch"]
    .rolling(15, min_periods=1)
    .mean()
)


# ============================================================
# 05. Summaries and transition matrices
# ============================================================

gate_order = ["accepted", "watch", "reroute", "fallback"]

def transition_matrix(labels, order):
    mat = pd.DataFrame(0.0, index=order, columns=order)
    labels = list(labels)
    for i in range(len(labels) - 1):
        mat.loc[labels[i], labels[i + 1]] += 1
    return mat.div(mat.sum(axis=1), axis=0).fillna(0)

predictive_transition_matrix = transition_matrix(
    stream["predictive_gate"],
    gate_order,
)

reactive_transition_matrix = transition_matrix(
    stream["reactive_gate"],
    ["accepted", "watch", "fallback"],
)

action_summary = (
    stream.groupby("predictive_action")
    .agg(
        windows=("window", "count"),
        mean_probability=("decompression_probability", "mean"),
        mean_pressure=("rolling_pressure", "mean"),
        mean_cgcs=("macro_cgcs_score", "mean"),
        mean_stability_gain=("stability_gain", "mean"),
        avoidable_fallback_rate=("avoidable_fallback", "mean"),
    )
    .reset_index()
    .sort_values("windows", ascending=False)
)

route_summary = (
    stream.groupby("macro_route")
    .agg(
        windows=("window", "count"),
        mean_probability=("decompression_probability", "mean"),
        mean_pressure=("rolling_pressure", "mean"),
        mean_cgcs=("macro_cgcs_score", "mean"),
        mean_stability_gain=("stability_gain", "mean"),
        predictive_reroute_rate=("predictive_action", lambda s: (s == "predictive_reroute").mean()),
        protective_fallback_rate=("predictive_action", lambda s: (s == "protective_fallback").mean()),
    )
    .reset_index()
    .sort_values("mean_probability", ascending=False)
)

summary = {
    "windows": int(N),
    "roc_auc": float(auc),
    "forecast_positive_windows": int(stream["forecast_decompression"].sum()),
    "actual_future_decompression_windows": int(stream["future_decompression"].sum()),
    "predictive_reroute_windows": int((stream["predictive_gate"] == "reroute").sum()),
    "reactive_fallback_windows": int((stream["reactive_gate"] == "fallback").sum()),
    "predictive_fallback_windows": int((stream["predictive_gate"] == "fallback").sum()),
    "avoidable_fallback_windows": int(stream["avoidable_fallback"].sum()),
    "mean_reactive_stability": float(stream["reactive_stability"].mean()),
    "mean_predictive_stability": float(stream["predictive_stability"].mean()),
    "mean_stability_gain": float(stream["stability_gain"].mean()),
    "mean_reactive_switch_rate": float(stream["reactive_switch_rate"].mean()),
    "mean_predictive_switch_rate": float(stream["predictive_switch_rate"].mean()),
}

summary_df = pd.DataFrame([summary])

classification_df = pd.DataFrame(
    classification_report(
        stream["future_decompression"],
        stream["forecast_decompression"],
        output_dict=True,
    )
).transpose()

feature_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": forecast_model.feature_importances_,
}).sort_values("importance", ascending=False)


# ============================================================
# 06. Save result artifacts
# ============================================================

routing_csv = "results/notebook23_predictive_constraint_routing.csv"
routing_json = "results/notebook23_predictive_constraint_routing.json"
summary_csv = "results/notebook23_summary.csv"
action_summary_csv = "results/notebook23_action_summary.csv"
route_summary_csv = "results/notebook23_route_summary.csv"
feature_importance_csv = "results/notebook23_feature_importance.csv"
predictive_transition_csv = "results/notebook23_predictive_gate_transition_matrix.csv"
reactive_transition_csv = "results/notebook23_reactive_gate_transition_matrix.csv"
classification_csv = "results/notebook23_forecast_classification_report.csv"

stream.to_csv(routing_csv, index=False)
summary_df.to_csv(summary_csv, index=False)
action_summary.to_csv(action_summary_csv, index=False)
route_summary.to_csv(route_summary_csv, index=False)
feature_importance.to_csv(feature_importance_csv, index=False)
predictive_transition_matrix.to_csv(predictive_transition_csv)
reactive_transition_matrix.to_csv(reactive_transition_csv)
classification_df.to_csv(classification_csv)

with open(routing_json, "w") as f:
    json.dump(stream.to_dict(orient="records"), f, indent=2)


# ============================================================
# 07. Figures
# ============================================================

# ------------------------------------------------------------
# Forecast probability timeline
# ------------------------------------------------------------

plt.figure(figsize=(16, 6))
plt.plot(windows, stream["decompression_probability"], label="forecast probability")
plt.axhline(0.50, linestyle="--", label="reroute threshold")
plt.axhline(0.70, linestyle="--", label="protective fallback threshold")
plt.title("Predictive Constraint Routing: Decompression Risk Timeline")
plt.xlabel("Window")
plt.ylabel("Forecast probability")
plt.legend()
risk_timeline_fig = "figures/notebook23_decompression_risk_timeline.png"
plt.savefig(risk_timeline_fig, bbox_inches="tight")
plt.close()

# ------------------------------------------------------------
# Reactive vs predictive gate timeline
# ------------------------------------------------------------

state_map = {"accepted": 3, "watch": 2, "reroute": 1, "fallback": 0}
reactive_map = {"accepted": 3, "watch": 2, "fallback": 0}

plt.figure(figsize=(16, 6))
plt.step(windows, [reactive_map[v] for v in stream["reactive_gate"]], where="post", label="reactive gate")
plt.step(windows, [state_map[v] for v in stream["predictive_gate"]], where="post", linestyle="--", label="predictive gate")
plt.yticks([0, 1, 2, 3], ["fallback", "reroute", "watch", "accepted"])
plt.title("Predictive Constraint Routing: Reactive vs Predictive Gate Timeline")
plt.xlabel("Window")
plt.ylabel("Gate")
plt.legend()
gate_timeline_fig = "figures/notebook23_reactive_vs_predictive_gate_timeline.png"
plt.savefig(gate_timeline_fig, bbox_inches="tight")
plt.close()

# ------------------------------------------------------------
# Stability comparison
# ------------------------------------------------------------

plt.figure(figsize=(16, 6))
plt.plot(windows, stream["reactive_stability"], label="reactive stability")
plt.plot(windows, stream["predictive_stability"], label="predictive stability")
plt.title("Predictive Constraint Routing: Stability Before vs After Forecast Routing")
plt.xlabel("Window")
plt.ylabel("Stability score")
plt.legend()
stability_fig = "figures/notebook23_reactive_vs_predictive_stability.png"
plt.savefig(stability_fig, bbox_inches="tight")
plt.close()

# ------------------------------------------------------------
# Switch-rate comparison
# ------------------------------------------------------------

plt.figure(figsize=(16, 6))
plt.plot(windows, stream["reactive_switch_rate"], label="reactive switch rate")
plt.plot(windows, stream["predictive_switch_rate"], label="predictive switch rate")
plt.title("Predictive Constraint Routing: Reactive vs Predictive Switch Rates")
plt.xlabel("Window")
plt.ylabel("Rolling switch rate")
plt.legend()
switch_fig = "figures/notebook23_reactive_vs_predictive_switch_rates.png"
plt.savefig(switch_fig, bbox_inches="tight")
plt.close()

# ------------------------------------------------------------
# Action counts
# ------------------------------------------------------------

plt.figure(figsize=(10, 6))
plt.bar(action_summary["predictive_action"], action_summary["windows"])
plt.xticks(rotation=35)
plt.title("Predictive Constraint Routing: Action Counts")
plt.ylabel("Window count")
action_counts_fig = "figures/notebook23_action_counts.png"
plt.savefig(action_counts_fig, bbox_inches="tight")
plt.close()

# ------------------------------------------------------------
# Stability gain by route
# ------------------------------------------------------------

plt.figure(figsize=(10, 6))
plt.bar(route_summary["macro_route"], route_summary["mean_stability_gain"])
plt.xticks(rotation=35)
plt.title("Predictive Constraint Routing: Mean Stability Gain by Macro Route")
plt.ylabel("Mean stability gain")
stability_gain_fig = "figures/notebook23_stability_gain_by_macro_route.png"
plt.savefig(stability_gain_fig, bbox_inches="tight")
plt.close()

# ------------------------------------------------------------
# Predictive transition matrix
# ------------------------------------------------------------

plt.figure(figsize=(7, 6))
plt.imshow(predictive_transition_matrix, aspect="auto")
plt.xticks(range(len(gate_order)), gate_order, rotation=35)
plt.yticks(range(len(gate_order)), gate_order)
plt.colorbar(label="Transition probability")
plt.title("Predictive Constraint Routing: Predictive Gate Transition Matrix")
plt.xlabel("Next gate")
plt.ylabel("Current gate")
predictive_transition_fig = "figures/notebook23_predictive_gate_transition_matrix.png"
plt.savefig(predictive_transition_fig, bbox_inches="tight")
plt.close()

# ------------------------------------------------------------
# Pressure vs risk and intervention
# ------------------------------------------------------------

plt.figure(figsize=(16, 6))
plt.plot(windows, stream["rolling_pressure"], label="rolling pressure")
plt.plot(windows, stream["decompression_probability"], label="decompression probability")
reroute_windows = stream[stream["predictive_gate"] == "reroute"]
fallback_windows = stream[stream["predictive_gate"] == "fallback"]
plt.scatter(reroute_windows["window"], reroute_windows["decompression_probability"], label="predictive reroute")
plt.scatter(fallback_windows["window"], fallback_windows["decompression_probability"], marker="x", label="protective fallback")
plt.title("Predictive Constraint Routing: Pressure, Risk, and Intervention")
plt.xlabel("Window")
plt.ylabel("Normalized score")
plt.legend()
pressure_risk_fig = "figures/notebook23_pressure_risk_intervention.png"
plt.savefig(pressure_risk_fig, bbox_inches="tight")
plt.close()

# ------------------------------------------------------------
# Avoided fallback summary
# ------------------------------------------------------------

avoid_summary = pd.DataFrame({
    "category": ["actual future decompression", "forecast positives", "avoidable fallback windows"],
    "windows": [
        int(stream["future_decompression"].sum()),
        int(stream["forecast_decompression"].sum()),
        int(stream["avoidable_fallback"].sum()),
    ],
})

plt.figure(figsize=(10, 6))
plt.bar(avoid_summary["category"], avoid_summary["windows"])
plt.xticks(rotation=25)
plt.title("Predictive Constraint Routing: Avoided Fallback Summary")
plt.ylabel("Window count")
avoided_fig = "figures/notebook23_avoided_fallback_summary.png"
plt.savefig(avoided_fig, bbox_inches="tight")
plt.close()


# ============================================================
# 08. Markdown report
# ============================================================

report_md = f"""
# Report 23 — Predictive Constraint Routing

Notebook 23 uses decompression forecasts to route before fallback occurs.

Constraint view:
> predictive routing is useful when forecast risk can stabilize compressed routes before fallback becomes necessary.

## Generated outputs

- Predictive constraint routing CSV: <a href="{routing_csv}">`{routing_csv}`</a>
- Predictive constraint routing JSON: <a href="{routing_json}">`{routing_json}`</a>
- Summary CSV: <a href="{summary_csv}">`{summary_csv}`</a>
- Action summary CSV: <a href="{action_summary_csv}">`{action_summary_csv}`</a>
- Route summary CSV: <a href="{route_summary_csv}">`{route_summary_csv}`</a>
- Feature importance CSV: <a href="{feature_importance_csv}">`{feature_importance_csv}`</a>
- Predictive gate transition matrix CSV: <a href="{predictive_transition_csv}">`{predictive_transition_csv}`</a>
- Reactive gate transition matrix CSV: <a href="{reactive_transition_csv}">`{reactive_transition_csv}`</a>
- Forecast classification report CSV: <a href="{classification_csv}">`{classification_csv}`</a>

- Figure: <a href="{risk_timeline_fig}">`{risk_timeline_fig}`</a>
- Figure: <a href="{gate_timeline_fig}">`{gate_timeline_fig}`</a>
- Figure: <a href="{stability_fig}">`{stability_fig}`</a>
- Figure: <a href="{switch_fig}">`{switch_fig}`</a>
- Figure: <a href="{action_counts_fig}">`{action_counts_fig}`</a>
- Figure: <a href="{stability_gain_fig}">`{stability_gain_fig}`</a>
- Figure: <a href="{predictive_transition_fig}">`{predictive_transition_fig}`</a>
- Figure: <a href="{pressure_risk_fig}">`{pressure_risk_fig}`</a>
- Figure: <a href="{avoided_fig}">`{avoided_fig}`</a>

## Summary

{summary_df.to_markdown(index=False)}

## Predictive action summary

{action_summary.to_markdown(index=False)}

## Macro route summary

{route_summary.to_markdown(index=False)}

## Feature importance

{feature_importance.to_markdown(index=False)}

## Predictive gate transition probabilities

{predictive_transition_matrix.to_markdown()}

## Reactive gate transition probabilities

{reactive_transition_matrix.to_markdown()}

## Forecast classification report

{classification_df.to_markdown()}

## Interpretation

- Predictive routing converts high decompression probability into route actions before fallback is observed.
- Predictive reroute windows act as intermediate states between watch and fallback.
- Protective fallback is reserved for high-risk, high-pressure windows.
- Stability gain estimates whether forecast-informed routing improves compressed-route reliability.
- Avoidable fallback windows identify where forecasting can reduce reactive collapse handling.

## Next step

Notebook 24 can build route-memory policy evaluation:
- compare routing policies across repeated trials,
- estimate stability distributions,
- score policy regret,
- select routing policies under CGCS-style constraints.
"""

report_path = "reports/notebook23_predictive_constraint_routing_report.md"
with open(report_path, "w") as f:
    f.write(report_md)


# ============================================================
# 09. Zip export
# ============================================================

zip_name = "notebook23_predictive_constraint_routing_outputs.zip"

with zipfile.ZipFile(zip_name, "w") as zipf:
    for folder in ["results", "figures", "reports"]:
        for root, _, files in os.walk(folder):
            for file in files:
                path = os.path.join(root, file)
                zipf.write(path)

print("Notebook 23 complete.")
print("ZIP:", zip_name)
print("Report:", report_path)


# ============================================================
# 10. Optional Colab download
# ============================================================

# from google.colab import files
# files.download(zip_name)
